# NLP Preprocessing Engine – Internship Assignment

This notebook implements a robust NLP preprocessing pipeline to clean noisy real-world text data.

## Task 1: Conceptual Understanding 
1. Difference between "Love" and "love" in NLP?
= In raw text, "Love" and "love" are different tokens due to case sensitivity.
Most NLP pipelines convert text to lowercase to avoid duplication.
Without normalization:
"Love" ≠ "love" → increases vocabulary size unnecessarily.

2. What happens if stopwords are not removed?
= The dataset becomes noisy and high-dimensional
Common words like the, is, and dominate frequency counts
Can reduce model performance (especially in models like TF-IDF)

3. When removing stopwords can be harmful
Sentiment Analysis
= "I am not happy" → removing "not" changes meaning completely
Question Answering / Chatbots
"What is your name?" → removing "what" breaks intent

4. Stemming vs Lemmatization
= Feature	Stemming	Lemmatization
Approach	Rule-based cutting	Dictionary-based
Output	"running" → "run" or "runn"	"running" → "run"
Accuracy	Less accurate	More accurate
Speed	Faster	Slower

## Task 2: Preprocessing Function

In [5]:
import re

def preprocess_text(text):
    if not isinstance(text, str) or text.strip() == "":
        return []

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove emails
    text = re.sub(r"\S+@\S+", "", text)

    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Lowercase
    text = text.lower()

    # Remove repeated characters (soooo → so)
    text = re.sub(r"(.)\1{2,}", r"\1", text)

    # Remove special characters/emojis
    text = re.sub(r"[^a-z\s]", "", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    tokens = text.split()

    # Remove short tokens except 'no', 'not'
    cleaned_tokens = [
        word for word in tokens 
        if len(word) > 2 or word in ["no", "not"]
    ]

    return cleaned_tokens

## Task 3: Stress Testing

In [10]:
test_sentences = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍😍",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://abcd.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

results = []

for sentence in test_sentences:
    tokens = preprocess_text(sentence)
    clean_sentence = " ".join(tokens)

    results.append({
        "original": sentence,
        "tokens": tokens,
        "clean_sentence": clean_sentence
    })

for r in results:
    print("\nOriginal:", r["original"])
    print("Tokens:", r["tokens"])
    print("Clean:", r["clean_sentence"])


Original: Get 100% FREE access now!!!
Tokens: ['get', 'free', 'access', 'now']
Clean: get free access now

Original: I absolutely looooved this product 😍😍
Tokens: ['absolutely', 'loved', 'this', 'product']
Clean: absolutely loved this product

Original: Worst service ever... 0/10
Tokens: ['worst', 'service', 'ever']
Clean: worst service ever

Original: Call me at 9876543210
Tokens: ['call']
Clean: call

Original: This is THE best course!!!
Tokens: ['this', 'the', 'best', 'course']
Clean: this the best course

Original: Visit https://abcd.com now!
Tokens: ['visit', 'now']
Clean: visit now

Original: Nooooo this is baaad!!!
Tokens: ['no', 'this', 'bad']
Clean: no this bad

Original: OK OK OK I got it
Tokens: ['got']
Clean: got

Original: Win $$$ now!!! Limited offer!!!
Tokens: ['win', 'now', 'limited', 'offer']
Clean: win now limited offer

Original: I am not happy with this
Tokens: ['not', 'happy', 'with', 'this']
Clean: not happy with this


## Task 4: Token Analytics

In [11]:
def token_analytics(tokens):
    if len(tokens) == 0:
        return 0, 0, 0

    total = len(tokens)
    unique = len(set(tokens))
    avg_len = sum(len(t) for t in tokens) / total

    return total, unique, round(avg_len, 2)

for r in results:
    total, unique, avg_len = token_analytics(r["tokens"])
    
    print("\nSentence:", r["original"])
    print("Total Tokens:", total)
    print("Unique Tokens:", unique)
    print("Avg Length:", avg_len)


Sentence: Get 100% FREE access now!!!
Total Tokens: 4
Unique Tokens: 4
Avg Length: 4.0

Sentence: I absolutely looooved this product 😍😍
Total Tokens: 4
Unique Tokens: 4
Avg Length: 6.5

Sentence: Worst service ever... 0/10
Total Tokens: 3
Unique Tokens: 3
Avg Length: 5.33

Sentence: Call me at 9876543210
Total Tokens: 1
Unique Tokens: 1
Avg Length: 4.0

Sentence: This is THE best course!!!
Total Tokens: 4
Unique Tokens: 4
Avg Length: 4.25

Sentence: Visit https://abcd.com now!
Total Tokens: 2
Unique Tokens: 2
Avg Length: 4.0

Sentence: Nooooo this is baaad!!!
Total Tokens: 3
Unique Tokens: 3
Avg Length: 3.0

Sentence: OK OK OK I got it
Total Tokens: 1
Unique Tokens: 1
Avg Length: 3.0

Sentence: Win $$$ now!!! Limited offer!!!
Total Tokens: 4
Unique Tokens: 4
Avg Length: 4.5

Sentence: I am not happy with this
Total Tokens: 4
Unique Tokens: 4
Avg Length: 4.0


## Task 5: Frequency Analysis

In [12]:
from collections import Counter

all_tokens = []
for r in results:
    all_tokens.extend(r["tokens"])

counter = Counter(all_tokens)

print("Top 10 Frequent Words:")
print(counter.most_common(10))

print("\nTop 5 Least Frequent Words:")
print(counter.most_common()[-5:])

Top 10 Frequent Words:
[('this', 4), ('now', 3), ('get', 1), ('free', 1), ('access', 1), ('absolutely', 1), ('loved', 1), ('product', 1), ('worst', 1), ('service', 1)]

Top 5 Least Frequent Words:
[('limited', 1), ('offer', 1), ('not', 1), ('happy', 1), ('with', 1)]


## Task 6: Full Pipeline

In [16]:
def full_pipeline(text_list):
    all_tokens = []
    clean_sentences = []

    for text in text_list:
        tokens = preprocess_text(text)
        all_tokens.extend(tokens)
        clean_sentences.append(" ".join(tokens))

    return {
        "tokens": all_tokens,
        "clean_sentences": clean_sentences
    }

## Task 7: Error Handling Testing

In [17]:
edge_cases = ["", "😂😂😂😂", "1234567890"]

for case in edge_cases:
    print("\nInput:", case)
    print("Output:", preprocess_text(case))


Input: 
Output: []

Input: 😂😂😂😂
Output: []

Input: 1234567890
Output: []
